# Linear Algebra with NumPy — Class Notes

**Goal:** Apply NumPy's linear algebra to vectors/matrices: dot and matrix products, transpose/reshape, norms, solving linear systems, and best practices.

## 0) Setup

In [ ]:

import numpy as np
rng = np.random.default_rng(123)
np.set_printoptions(precision=3, suppress=True)

## 1) Dot (scalar) product

For vectors \(c, v \in \mathbb{R}^n\):  
\(\displaystyle c\cdot v = \sum_{i=1}^{n} c_i v_i\)

In NumPy:
- `a @ b` (preferred)
- `np.dot(a, b)`

In [ ]:

a = np.array([1,2,3], dtype=float)
b = np.array([4,5,6], dtype=float)
print("a@b:", a @ b)
print("np.dot:", np.dot(a,b))

## 2) Matrix product (2-D @ 2-D)

If `A` is `(m, p)` and `B` is `(p, n)`, then `C = A @ B` is `(m, n)`.

In [ ]:

A = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)   # (2,3)
B = np.array([[1,  0],
              [0,  1],
              [1, -1]], dtype=float)      # (3,2)
C = A @ B
print("C shape:", C.shape, "
C=
", C)

## 3) Transpose & Reshape

- Transpose 2-D: `A.T`
- General axis permute: `np.transpose(A, axes=...)`
- Reshape without changing data: `A.reshape(new_shape)`

In [ ]:

M = np.arange(1,13).reshape(3,4)
print("M:
", M)
print("M.T:
", M.T)

T = np.transpose(M.reshape(2,2,3), axes=(0,2,1))
print("permute axes shape:", T.shape)

v = np.arange(12)
print("v reshape (3,4):
", v.reshape(3,4))

## 4) Norms & vector length

- `np.linalg.norm(x)` → Euclidean (L2) norm by default.
- `ord=1` (L1), `ord=np.inf` (max). For matrices, consider `axis` with `keepdims=True`.

In [ ]:

x = np.array([3.0, 4.0])
print("L2:", np.linalg.norm(x))
print("L1:", np.linalg.norm(x, ord=1))
print("L_inf:", np.linalg.norm(x, ord=np.inf))

V = rng.normal(size=(3,4))
row_norms = np.linalg.norm(V, axis=1, keepdims=True)
print("row norms:
", row_norms)
print("row-normalized:
", V/row_norms)

## 5) Solve linear systems (avoid explicit inverse)

Solve \(Ax=b\) with `np.linalg.solve(A, b)` when `A` is square and full rank.  
Use least squares for non-square: `np.linalg.lstsq(A, b, rcond=None)`.

In [ ]:

A = np.array([[3., 2.],
              [1., 2.]])
b = np.array([18., 14.])

x = np.linalg.solve(A, b)
print("x:", x)

# Least squares (overdetermined)
A_ls = np.array([[1., 0.],
                 [1., 1.],
                 [1., 2.]])
b_ls = np.array([1., 2., 2.])
x_ls, *_ = np.linalg.lstsq(A_ls, b_ls, rcond=None)
print("x_ls (least squares):", x_ls)

## 6) Determinant & inverse (use with care)

- `np.linalg.det(A)` for determinant.
- `np.linalg.inv(A)` computes inverse — avoid in numeric pipelines; prefer `solve`.

**Tip:** Nearly singular matrices (ill-conditioned) lead to unstable inverses.

In [ ]:

A = np.array([[1., 2.],
              [3., 4.]])
print("det(A):", np.linalg.det(A))
print("inv(A):
", np.linalg.inv(A))

# Better: solve A x = b
b = np.array([1., 0.])
x = np.linalg.solve(A, b)
print("solve(A,b):", x)

## 7) Eigen / SVD (very brief intro)

- Eigenvalues/vectors: `np.linalg.eig(A)` for square `A`.
- SVD: `U, S, Vt = np.linalg.svd(M, full_matrices=False)` — robust for many tasks.

In [ ]:

A = np.array([[2., 0.],
              [0., 1.]])
w, V = np.linalg.eig(A)
print("eigvals:", w)
print("eigvecs:
", V)

M = rng.normal(size=(4,3))
U, S, Vt = np.linalg.svd(M, full_matrices=False)
print("SVD shapes:", U.shape, S.shape, Vt.shape)

### Quick self-checks (asserts)

In [ ]:

# Dot/matmul shape and values
a = np.array([1,2,3]); b = np.array([4,5,6])
assert a @ b == 32

A = np.array([[1,2],[3,4.]], dtype=float)
I = np.eye(2)
assert np.allclose(A @ np.linalg.inv(A), I, atol=1e-7)

# Solve correctness
b = np.array([5., 11.])
x = np.linalg.solve(A, b)
assert np.allclose(A @ x, b)

# Peer-Instruction Cards (Linear Algebra)

Non-coding tasks: predict / pick / minimal fix.

### Card LA-1 — Shape predict
If `A.shape == (2,3)` and `B.shape == (3,4)`, what is `(A @ B).shape`?  
Is `B @ A` valid? Why/why not?

### Card LA-2 — Dot vs elementwise
```python
a = np.array([1,2,3])
b = np.array([4,5,6])
print(a*b)   # ?
print(a@b)   # ?
```
Predict both outputs.

### Card LA-3 — Transpose gotcha
For 1-D arrays `x`, `x.T is x`. Why? What shapes make transpose matter?

### Card LA-4 — Minimal fix (solve vs inverse)
Intent: solve `Ax=b`. Which is numerically safer? Circle one and explain.
```python
x = np.linalg.inv(A) @ b
x = np.linalg.solve(A, b)
```

### Card LA-5 — Rank intuition (least squares)
Overdetermined `A` (3×2) and `b` length 3. Circle the correct solver and explain in one line:
`np.linalg.solve(A,b)` vs `np.linalg.lstsq(A,b,rcond=None)`.

## Mini Challenges

1) **Cosine similarity:** Given `x, y` 1-D, compute cosine similarity \(\frac{x\cdot y}{\|x\|\|y\|}\).  
2) **Projection:** Project vector `x` onto `u` (non-zero) using \(\text{proj}_u(x)=\frac{x\cdot u}{u\cdot u}u\).  
3) **Batch matmul:** Given `X` shape `(N, d)` and weight `W` shape `(d, k)`, compute `Y = X @ W` and confirm shape `(N, k)`.

In [ ]:

# Reference checks
x = np.array([1.,2.,3.]); y = np.array([4.,5.,6.])
cos = (x @ y) / (np.linalg.norm(x)*np.linalg.norm(y))
print("1) cosine:", round(float(cos), 6))

u = np.array([2.,0.,0.])
proj = ((x @ u) / (u @ u)) * u
print("2) proj:", proj)

N, d, k = 5, 4, 3
X = rng.normal(size=(N,d))
W = rng.normal(size=(d,k))
Y = X @ W
print("3) Y shape:", Y.shape)

## Takeaways
- Prefer `@`/`matmul` for matrix products; `*` is elementwise.
- Use `solve`/`lstsq` rather than explicit inverses in pipelines.
- Transpose and reshape change **views of shape**, not data values.
- Check dimensions early: write shapes next to symbols on the board.
- For stability, be mindful of ill-conditioned matrices.